In [0]:
#%pip install --upgrade databricks-langchain langchain-community langchain databricks-sql-connector
#%pip install --upgrade langchain langchain-core langchain-community
#%pip install databricks-vectorsearch
#dbutils.library.restartPython() # Restarts the Python process in a Databricks notebook

from databricks.vector_search.client import VectorSearchClient

# Initialize the client (automatically authenticates in a Databricks notebook)
vsc = VectorSearchClient(workspace_url="https://dbc-1e25575d-1d41.cloud.databricks.com/")

# Get the index object (replace with your catalog.schema.index_name and endpoint name)
# Best practice is to initialize the index object once and reuse it across queries
index  = vsc.get_index(
    endpoint_name="vector_index_endpoint",
    index_name="workspace.default.rag_sales_data"
)

# Perform a similarity search
query_text = "How can I find large account ?"
num_results = 1 # Retrieve between 10-100 results for best performance

results = index.similarity_search(
    query_text=query_text,
    columns=["combined_text"], # Specify columns to return
    filters={"YEAR_ID": 2003},
    num_results=num_results
)
results["result"]["data_array"][0][0]
 
from langchain_community.vectorstores import DatabricksVectorSearch
from databricks_langchain import ChatDatabricks
vectorstore = DatabricksVectorSearch(index)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# --- Setup LLM ---
# Use a Databricks-hosted LLM (e.g., Claude 3 Sonnet on Databricks)
from databricks_langchain import ChatDatabricks

# Initialize the chat model with a served endpoint
llm = ChatDatabricks(
    endpoint="databricks-gemma-3-12b", # Example endpoint name
    temperature=0.1,
    max_tokens=256
)

# --- Build the RAG Chain ---
# Create a QA chain that uses the retriever to fetch context
from langchain_classic.chains import RetrievalQA 
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff", # 'stuff' combines all retrieved docs into one prompt
    retriever=retriever,
    return_source_documents=True # To see the retrieved snippets
)

# --- Query the System ---
question=dbutils.widgets.text("Question", "How can I find large account ?", "Question:") 
 
response = qa_chain.invoke({"query": dbutils.widgets.get("Question")})
print("Question:")
print(dbutils.widgets.get("Question"))

print("Answer:")
print(response['result'])



[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.


/home/spark-4252ff80-6504-4bee-a218-c4/.ipykernel/3419/command-4882349176686394-2274253467:32: LangChainDeprecationWarning: The class `DatabricksVectorSearch` was deprecated in LangChain 0.3.3 and will be removed in 1.0. An updated version of the class exists in the `databricks-langchain package and should be used instead. To use it run `pip install -U `databricks-langchain` and import as `from `databricks_langchain import DatabricksVectorSearch``.
  vectorstore = DatabricksVectorSearch(index)


[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
Question:
who are the best sellers
Answer:
Based on the provided information, here's a ranking of the best sellers:

1.  **Mini Gifts Distributors Ltd.** - $5720.75
2.  **Scandinavian Gift Ideas** - $3363.52
3.  **AV Stores, Co.** - $2700.00
